# Linear Classifier & Perceptron

---

**Structure**
1. Introduction — what the algorithm does and the intuition behind it
2. The Math — key equations
3. Problem Class — when and where to use it
4. Implementation — applied to the Breast Cancer Wisconsin dataset
5. Results — what the numbers tell us
6. Limitations — where the algorithm breaks down

## 1. Introduction

A **linear classifier** is the simplest supervised learning model for binary classification. The core idea is to find a straight line (in 2D), plane (in 3D), or more generally a **hyperplane** in high dimensions that separates two classes of data.

Every point in the feature space falls on one side of this hyperplane and gets assigned a label accordingly — either $+1$ or $-1$.

**Intuition:**  
Imagine plotting patients on a chart where each axis is a medical measurement. A linear classifier tries to draw a single straight line that puts all malignant tumours on one side and all benign tumours on the other.

**The Perceptron** is the simplest algorithm for *learning* that hyperplane. It works by scanning through the training data and whenever it makes a mistake — it nudges the boundary in the right direction. Given enough passes through the data, if a separating hyperplane exists, the Perceptron is guaranteed to find one.

**The SVM (Support Vector Machine)** goes a step further: instead of just finding *any* separating hyperplane, it finds the one with the **largest margin** — the widest gap between the two classes. This tends to generalise better to unseen data.

## 2. The Math

### Classifier
Given a feature vector $x \in \mathbb{R}^d$, a weight vector $\theta \in \mathbb{R}^d$, and a bias $\theta_0 \in \mathbb{R}$:

$$h(x;\,\theta,\theta_0) = \text{sign}(\theta \cdot x + \theta_0)$$

The **decision boundary** is the hyperplane where $\theta \cdot x + \theta_0 = 0$.  
The signed distance from any point $x_0$ to the boundary is $\dfrac{\theta \cdot x_0 + \theta_0}{\|\theta\|}$.

---

### Training Error
The fraction of training examples the classifier gets wrong:

$$\varepsilon_n(\theta, \theta_0) = \frac{1}{n}\sum_{i=1}^{n} \mathbf{1}\bigl\{y^{(i)}(\theta \cdot x^{(i)} + \theta_0) \leq 0\bigr\}$$

The product $y^{(i)}(\theta \cdot x^{(i)} + \theta_0)$ is called the **agreement** — positive means correct, negative means wrong.

---

### Perceptron Update
On each misclassified example $(x^{(i)}, y^{(i)})$:

$$\theta \leftarrow \theta + y^{(i)} x^{(i)}, \qquad \theta_0 \leftarrow \theta_0 + y^{(i)}$$

This rotates the hyperplane towards the misclassified point.

---

### Hinge Loss & SVM Objective
The **hinge loss** measures how far a prediction is from being confidently correct:

$$\text{Loss}_h(z) = \max(0,\; 1 - z) \quad \text{where } z = y^{(i)}(\theta \cdot x^{(i)} + \theta_0)$$

The SVM minimises the average hinge loss plus an L2 regulariser that maximises the margin:

$$J(\theta, \theta_0) = \frac{1}{n}\sum_{i=1}^{n} \text{Loss}_h(z^{(i)}) + \frac{\lambda}{2}\|\theta\|^2$$

The **margin width** between the two class boundaries is $\dfrac{2}{\|\theta\|}$. Minimising $\|\theta\|^2$ (via $\lambda$) pushes this margin wider.

---

### Stochastic Gradient Descent
Pick a random example $i$, then update:

$$\theta \leftarrow \theta - \eta \nabla_\theta \left[\text{Loss}_h(z^{(i)}) + \frac{\lambda}{2}\|\theta\|^2\right]$$

Where $\eta > 0$ is the learning rate. The gradient of the hinge loss is:

$$\nabla_\theta \text{Loss}_h = \begin{cases} -y^{(i)} x^{(i)} & \text{if } z < 1 \\ 0 & \text{if } z \geq 1 \end{cases}$$

## 3. Problem Class

**Linear classifiers are well-suited for:**
- Binary classification tasks (two classes)
- Data that is **linearly separable** or close to it
- **High-dimensional sparse data** — e.g. text classification (bag of words), where the number of features can be enormous but most are zero
- Problems where **interpretability matters** — the weight vector $\theta$ directly tells you which features matter most and in which direction
- Large datasets where computational efficiency is critical — SGD updates are $O(d)$ per step

**Typical domains:**
- Medical diagnosis (tumour classification, disease prediction)
- Spam filtering
- Sentiment analysis
- Credit risk scoring

**Not well-suited for:**
- Data with non-linear decision boundaries (use kernels or neural networks)
- Multi-class problems without modification (requires one-vs-rest or one-vs-one)
- Heavily overlapping classes with no separating margin

---
## 4. Implementation
### Dataset: Breast Cancer Wisconsin

569 tumour samples, 30 numeric features (radius, texture, perimeter, area, smoothness, …), binary label: **malignant** (−1) or **benign** (+1).

Source: [UCI Machine Learning Repository — Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) · also available via `sklearn.datasets.load_breast_cancer()`

All algorithms train on the full 30-dimensional feature space. PCA is used only to project down to 2D for visualisation.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# ── Load & prepare ────────────────────────────────────────────────────────────
data = load_breast_cancer()
X_raw, y_raw = data.data, data.target

y = np.where(y_raw == 1, 1, -1)           # benign=+1, malignant=-1

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pca = PCA(n_components=2, random_state=42)
X_vis = pca.fit_transform(X_train)        # 2D projection for visualisation only

print(f'Features  : {X.shape[1]}')
print(f'Train     : {X_train.shape[0]}  ({(y_train==1).sum()} benign, {(y_train==-1).sum()} malignant)')
print(f'Test      : {X_test.shape[0]}')
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}')

# ── Shared helpers ────────────────────────────────────────────────────────────
def linear_classifier(X, theta, theta0=0.0):
    return np.sign(X @ theta + theta0)

def error_rate(X, y, theta, theta0=0.0):
    return np.mean(linear_classifier(X, theta, theta0) != y)

def hinge_loss(z):
    return np.maximum(0, 1 - z)

def plot_2d(X2, y, theta_2d=None, theta0=0.0, title='', margin=False):
    fig, ax = plt.subplots(figsize=(7, 5))
    colors = {1: 'steelblue', -1: 'tomato'}
    labels = {1: 'Benign (+1)', -1: 'Malignant (−1)'}
    for lbl in [1, -1]:
        m = y == lbl
        ax.scatter(X2[m, 0], X2[m, 1], c=colors[lbl], label=labels[lbl],
                   edgecolors='k', s=40, alpha=0.7, zorder=3)
    if theta_2d is not None and abs(theta_2d[1]) > 1e-10:
        x1 = np.linspace(X2[:, 0].min() - 1, X2[:, 0].max() + 1, 300)
        ax.plot(x1, -(theta0 + theta_2d[0]*x1)/theta_2d[1], 'k-', lw=2, label='Decision boundary')
        if margin:
            nrm = np.linalg.norm(theta_2d)
            ax.plot(x1, -(theta0+1+theta_2d[0]*x1)/theta_2d[1], 'k--', lw=1.2, label=f'Margin (width={2/nrm:.2f})')
            ax.plot(x1, -(theta0-1+theta_2d[0]*x1)/theta_2d[1], 'k--', lw=1.2)
    ax.set_xlabel('PCA 1'); ax.set_ylabel('PCA 2')
    ax.set_title(title); ax.legend(loc='upper right')
    plt.tight_layout(); plt.show()

**Observation**

The training set has 285 benign vs 170 malignant — a 63/37 split. That imbalance matters: a model that blindly predicts every tumour as benign would already hit ~63% accuracy without learning anything. Error rate alone won't tell the full story here.

62.7% of variance captured in just 2 components out of 30 is a strong signal that the data has a clear dominant structure. The classes are likely well-separated along these directions, which we'll see in the next plot.

### 4.1 Explore the Data

In [ ]:
plot_2d(X_vis, y_train, title='Training set — PCA 2D projection')

**Observation — Data plot**

Even in this lossy 2D projection the two classes separate quite cleanly — the benign cluster sits in the upper-left and the malignant cluster in the lower-right, with only a modest overlap region in between. This is a strong signal that a linear classifier has a real shot at doing well here.

It is also worth noting there is no clean gap — the classes overlap, so some misclassifications are inevitable. In a medical context, misclassifying malignant as benign (a false negative) is far more costly than the reverse, something the algorithms below don't account for directly.

### 4.2 Perceptron

In [ ]:
def perceptron(X, y, X_te, y_te, T=20):
    n, d = X.shape
    theta, theta0 = np.zeros(d), 0.0
    tr_hist, te_hist = [], []
    for _ in range(T):
        for i in range(n):
            if y[i] * (theta @ X[i] + theta0) <= 0:
                theta  += y[i] * X[i]
                theta0 += y[i]
        tr_hist.append(error_rate(X,    y,    theta, theta0))
        te_hist.append(error_rate(X_te, y_te, theta, theta0))
    return theta, theta0, tr_hist, te_hist

theta_p, theta0_p, tr_p, te_p = perceptron(X_train, y_train, X_test, y_test, T=20)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tr_p, label='Train', marker='o')
axes[0].plot(te_p, label='Test',  marker='s', linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Error rate')
axes[0].set_title('Perceptron — Error per epoch')
axes[0].legend(); axes[0].grid(True, linestyle='--', alpha=0.5)

theta_p_2d = pca.components_ @ theta_p
x1 = np.linspace(X_vis[:, 0].min()-1, X_vis[:, 0].max()+1, 300)
for lbl, col, lbl_str in [(1,'steelblue','Benign (+1)'),(-1,'tomato','Malignant (−1)')]:
    m = y_train == lbl
    axes[1].scatter(X_vis[m,0], X_vis[m,1], c=col, label=lbl_str, edgecolors='k', s=40, alpha=0.7)
if abs(theta_p_2d[1]) > 1e-10:
    axes[1].plot(x1, -(theta0_p + theta_p_2d[0]*x1)/theta_p_2d[1], 'k-', lw=2, label='Decision boundary')
axes[1].set_title('Perceptron — Decision boundary (PCA 2D)')
axes[1].set_xlabel('PCA 1'); axes[1].set_ylabel('PCA 2'); axes[1].legend()

plt.tight_layout(); plt.show()
print(f'Perceptron — train err: {tr_p[-1]:.2%}   test err: {te_p[-1]:.2%}')

**Observation — Perceptron**

The error curve drops sharply in the first 2–3 epochs and then levels off — the algorithm finds a working boundary very quickly. This is characteristic of near-linearly-separable data: there are not many corrections needed before the weights point in roughly the right direction.

Notice the gap between train and test error. The Perceptron keeps updating as long as it makes any training mistake, so it tends to overfit slightly to the specific examples it last corrected on. The boundary it settles on is essentially arbitrary — wherever it happened to stop — rather than the best possible boundary for unseen data.

Looking at the decision boundary plot: it cuts through the PCA space at a reasonable angle, but it sits relatively close to some of the malignant points. There is no explicit push to move it away from the data — that is the key weakness the SVM is designed to fix.

### 4.3 SVM with SGD

In [ ]:
def svm_sgd(X, y, X_te, y_te, lam=0.01, eta=0.005, T=100, seed=0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    theta, theta0 = np.zeros(d), 0.0
    obj_hist, tr_hist, te_hist = [], [], []
    for t in range(T * n):
        i = rng.integers(n)
        z = y[i] * (theta @ X[i] + theta0)
        if z < 1:
            theta  -= eta * (-y[i] * X[i] + lam * theta)
            theta0 -= eta * (-y[i])
        else:
            theta  -= eta * lam * theta
        if t % n == 0:
            z_all = y * (X @ theta + theta0)
            obj_hist.append(np.mean(hinge_loss(z_all)) + (lam/2) * theta @ theta)
            tr_hist.append(error_rate(X,    y,    theta, theta0))
            te_hist.append(error_rate(X_te, y_te, theta, theta0))
    return theta, theta0, obj_hist, tr_hist, te_hist

theta_svm, theta0_svm, obj_hist, tr_svm, te_svm = svm_sgd(
    X_train, y_train, X_test, y_test, lam=0.01, eta=0.005, T=100
)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(obj_hist, color='darkorange', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Objective J(θ, θ₀)')
axes[0].set_title('SVM — Objective convergence')
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].plot(tr_svm, label='Train', marker='o', markersize=3)
axes[1].plot(te_svm, label='Test',  marker='s', markersize=3, linestyle='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Error rate')
axes[1].set_title('SVM — Train vs Test error')
axes[1].legend(); axes[1].grid(True, linestyle='--', alpha=0.5)

theta_svm_2d = pca.components_ @ theta_svm
x1 = np.linspace(X_vis[:,0].min()-1, X_vis[:,0].max()+1, 300)
for lbl, col, lbl_str in [(1,'steelblue','Benign (+1)'),(-1,'tomato','Malignant (−1)')]:
    m = y_train == lbl
    axes[2].scatter(X_vis[m,0], X_vis[m,1], c=col, label=lbl_str, edgecolors='k', s=40, alpha=0.7)
if abs(theta_svm_2d[1]) > 1e-10:
    nrm = np.linalg.norm(theta_svm_2d)
    axes[2].plot(x1, -(theta0_svm+theta_svm_2d[0]*x1)/theta_svm_2d[1], 'k-', lw=2, label='Boundary')
    axes[2].plot(x1, -(theta0_svm+1+theta_svm_2d[0]*x1)/theta_svm_2d[1], 'k--', lw=1.2, label=f'Margin (w={2/nrm:.2f})')
    axes[2].plot(x1, -(theta0_svm-1+theta_svm_2d[0]*x1)/theta_svm_2d[1], 'k--', lw=1.2)
axes[2].set_title('SVM — Decision boundary + margin (PCA 2D)')
axes[2].set_xlabel('PCA 1'); axes[2].set_ylabel('PCA 2'); axes[2].legend()

plt.tight_layout(); plt.show()
print(f'SVM — train err: {tr_svm[-1]:.2%}   test err: {te_svm[-1]:.2%}   margin: {2/np.linalg.norm(theta_svm):.4f}')

**Observation — SVM with SGD**

The objective curve descends steadily but is noticeably noisy — this is expected with SGD since each update is based on a single random example rather than the full dataset. It does not converge to a flat line; it oscillates around a minimum. In practice you would run for enough epochs that the oscillation is small relative to the initial drop.

The train and test error curves are much closer together than with the Perceptron, which is the key benefit of the margin: by explicitly pushing the boundary away from both classes, the SVM finds a boundary that is more robust to examples it hasn't seen.

In the boundary plot, the dashed margin lines are visibly separated from the bulk of each class. Points that fall inside or across the margin are the "support vectors" — the only examples that actually influence the final boundary. The rest of the training data is irrelevant once the SVM is trained.

### 4.4 Effect of Regularisation (λ)

In [ ]:
lambdas = [0.0001, 0.01, 0.1, 1.0]
results = []
for lam in lambdas:
    th, th0, _, tr, te = svm_sgd(X_train, y_train, X_test, y_test, lam=lam, eta=0.005, T=100)
    results.append((lam, tr[-1], te[-1], 2/np.linalg.norm(th)))
    print(f'λ={lam:<6}  train={tr[-1]:.2%}  test={te[-1]:.2%}  margin={2/np.linalg.norm(th):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(lambdas)); w = 0.35
lam_labels = [str(l) for l in lambdas]

axes[0].bar(x-w/2, [r[1] for r in results], w, label='Train', color='steelblue', alpha=0.8)
axes[0].bar(x+w/2, [r[2] for r in results], w, label='Test',  color='tomato',    alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(lam_labels)
axes[0].set_xlabel('λ'); axes[0].set_ylabel('Error rate')
axes[0].set_title('Error rate vs λ'); axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.4, axis='y')

axes[1].bar(x, [r[3] for r in results], color='darkorange', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(lam_labels)
axes[1].set_xlabel('λ'); axes[1].set_ylabel('Margin width (2/‖θ‖)')
axes[1].set_title('Margin width vs λ')
axes[1].grid(True, linestyle='--', alpha=0.4, axis='y')

plt.tight_layout(); plt.show()

**Observation — Effect of λ**

The two charts tell the classic regularisation story clearly.

In the error chart: at very low λ (0.0001) the training error is minimal but the test error is slightly elevated — the model has memorised the training set too closely and doesn't generalise as well. As λ increases the training error rises, but the test error initially *drops* before rising again at λ=1.0. The sweet spot where both are lowest is around λ=0.01.

In the margin chart: margin width grows monotonically with λ, which makes sense mathematically — a larger penalty on ‖θ‖ forces the weights to be smaller, which directly widens the margin (2/‖θ‖). But a very wide margin means the boundary can't fit tightly enough to the data, causing more errors.

The key practical takeaway: λ is not a knob you set once — it needs to be tuned, typically via cross-validation, because the right value depends on how separable the data is.

---
## 5. Results

| Method | Train error | Test error |
|---|---|---|
| Perceptron | ~2–5% | ~5–10% |
| SVM (λ=0.01) | ~2–4% | ~3–6% |

**What the numbers tell us:**
- Both algorithms work well on this dataset — the two classes are largely linearly separable in the standardised feature space
- The Perceptron converges quickly (within ~5 epochs) but the boundary it finds is arbitrary — it stops as soon as there are no more mistakes, not necessarily at the best boundary
- The SVM finds a wider-margin boundary which tends to generalise slightly better to the test set
- At λ=0.0001 (very little regularisation) the SVM overfits slightly — train error is low but test error is higher than at λ=0.01
- At λ=1.0 (heavy regularisation) the margin is wide but accuracy drops — the model is too constrained to fit the data well
- The sweet spot on this dataset is around λ=0.01, where train and test error are both minimised

---
## 6. Limitations

**Perceptron**
- Only guaranteed to converge if the data is **linearly separable** — on noisy real data it may cycle forever
- The boundary it finds depends on the order examples are seen and the initialisation — there is no unique solution
- Has no notion of margin — it accepts any separating hyperplane, even one very close to the data

**Linear classifiers generally**
- Cannot learn **non-linear decision boundaries** — if the true boundary is curved, a linear model will always underfit regardless of how much data you have
- Sensitive to **feature scaling** — features with large ranges dominate the dot product, which is why standardisation is essential
- **Binary only** by default — multi-class requires one-vs-rest or one-vs-one wrappers, which adds complexity
- The **Perceptron convergence theorem** only holds for linearly separable data; real datasets are rarely perfectly separable

**SVM with SGD**
- The quality of the solution depends heavily on the choice of **learning rate η** and **regularisation λ** — both require tuning
- SGD introduces noise; the objective oscillates rather than converging smoothly, making it harder to know when to stop
- Does not produce **probability estimates** natively — you only get a hard class label, not a confidence score